In [0]:
# =============================================================================
# NOTEBOOK  : silver_warehouse_inventory
# PURPOSE   : bronze.warehouse_inventory → silver.warehouse_inventory
# LAYER     : Silver (clean, validate, recompute, merge)
# FREQUENCY : Hourly
# PATTERN   : spark.read + audit watermark (production pattern)
#
# TRANSFORM:
#   - last_updated     : String → TimestampType
#   - available_stock  : recomputed as current_stock - reserved_stock
#                        source derivation not trusted
#   - stock columns    : already BIGINT from source, keep as-is
#
# MERGE & DEDUP LOGIC (SCD Type 1 — latest snapshot wins):
#   Merge key : warehouse_id + product_id
#   Dedup     : window on warehouse_id + product_id ordered by ingested_at DESC
#               keep most recently ingested snapshot per pair
#               hourly file may have multiple snapshots for same pair
#
#   Case 1 : warehouse_id + product_id NOT in silver → INSERT
#   Case 2 : pair IN silver, any stock value changed  → UPDATE all stock cols
#   Case 3 : pair IN silver, no change               → IGNORE
#
# NOTE: SCD Type 1 — silver always reflects CURRENT state
#       No history kept — old values overwritten on update
#       Historical stock levels available via Delta time travel if needed
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_WAREHOUSE_INVENTORY_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_timestamp, row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_warehouse_inventory"]
TARGET_TABLE = cfg["tables"]["silver_warehouse_inventory"]
PIPELINE     = "silver_warehouse_inventory"

In [0]:
# ── Read + Transform + Merge ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # ── INCREMENTAL READ ──────────────────────────────────────────────────────
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = spark.read.table(SOURCE_TABLE)
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new bronze rows since last run")
 
    if rows_read == 0:
        print(f"[SKIP] No new rows to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0,
                  extra={"rows_inserted": "0", "rows_updated": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
 
        # 1. last_updated: String → TimestampType
        .withColumn("last_updated",
            to_timestamp(col("last_updated")))
 
        # 2. Recompute available_stock — never trust source derivation
        #    Can be negative — over-reservation scenario, allowed
        .withColumn("available_stock",
            col("current_stock") - col("reserved_stock"))
 
        # 3. Silver audit columns
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("warehouse_inventory_parquet"))
        # ingested_at carried from bronze automatically via .select()
    )
 
    # ── DEDUP ─────────────────────────────────────────────────────────────────
    # Keep most recently ingested snapshot per warehouse_id + product_id
    # Done BEFORE .select() so ingested_at available as tiebreaker
    window = (
        Window
        .partitionBy("warehouse_id", "product_id")
        .orderBy(col("ingested_at").desc())
    )
 
    df = (
        df
        .withColumn("_row_num", row_number().over(window))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
    )
 
    rows_after_dedup = df.count()
    if rows_read != rows_after_dedup:
        print(f"[DEDUP] kept latest snapshot, "
              f"dropped {rows_read - rows_after_dedup} older snapshots")
 
    # Enforce silver schema — done AFTER dedup so ingested_at was available
    df = df.select([f.name for f in SILVER_WAREHOUSE_INVENTORY_SCHEMA.fields])
 
    # ── MERGE INTO SILVER (SCD Type 1) ────────────────────────────────────────
    # Case 2: any stock value changed → UPDATE all stock columns
    # Case 1: new pair → INSERT
    # Case 3: no change → no rule fires → IGNORE
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            df.alias("s"),
            """
            t.warehouse_id = s.warehouse_id AND
            t.product_id   = s.product_id
            """
        )
        .whenMatchedUpdate(
            condition="""
                t.current_stock   != s.current_stock   OR
                t.reserved_stock  != s.reserved_stock  OR
                t.available_stock != s.available_stock OR
                t.reorder_level   != s.reorder_level   OR
                t.max_stock       != s.max_stock       OR
                t.location_zone   != s.location_zone
            """,
            set={
                "current_stock":   "s.current_stock",
                "reserved_stock":  "s.reserved_stock",
                "available_stock": "s.available_stock",
                "reorder_level":   "s.reorder_level",
                "max_stock":       "s.max_stock",
                "last_updated":    "s.last_updated",
                "location_zone":   "s.location_zone",
                "processed_at":    "current_timestamp()",
                "pipeline_run_id": f"'{run_id}'"
                # ingested_at NOT updated — preserve original bronze landing time
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[DONE] inserted={rows_inserted} | updated={rows_updated}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read,
        rows_written=rows_inserted + rows_updated,
        extra={
            "rows_inserted": str(rows_inserted),
            "rows_updated":  str(rows_updated)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())
 
# 3. Nulls in key columns
print("Null key columns:",
    spark.read.table(TARGET_TABLE)
    .filter(col("warehouse_id").isNull() | col("product_id").isNull())
    .count()
)
 
# 4. Verify available_stock = current - reserved
print("Rows where available_stock != current - reserved (should be 0):",
    spark.read.table(TARGET_TABLE)
    .filter(
        col("available_stock") !=
        col("current_stock") - col("reserved_stock")
    )
    .count()
)
 
# 5. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)